# Okapi BM25

**BM25** ("Best Matching 25") is a ranking function that scores how relevant a document is to a search query. It is the modern successor to TF-IDF and the default relevance model in search engines such as Elasticsearch and Lucene.

Like TF-IDF, BM25 rewards terms that are **frequent in a document** but **rare across the corpus**. What it adds are fixes for two weaknesses of raw TF-IDF:

1. **Term-frequency saturation.** A word appearing 100 times in a document is not 100× more relevant than appearing once. BM25 lets term frequency *saturate* — each additional occurrence contributes less than the last. The rate of saturation is controlled by $k_1$.
2. **Document-length normalization.** Long documents naturally contain more occurrences of any term, which would unfairly inflate their scores. BM25 discounts a document's length relative to the corpus average. The strength of this correction is controlled by $b$.

### The scoring function

For a query $q$ against a document $d$, BM25 sums a contribution from each query term $t$:

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f_{t,d} \,(k_1 + 1)}{f_{t,d} + k_1\left(1 - b + b \cdot \dfrac{|d|}{\text{avgdl}}\right)}$$

where $f_{t,d}$ is the count of term $t$ in document $d$, $|d|$ is the document's length in **tokens**, and $\text{avgdl}$ is the average document length across the corpus.

### Term-frequency saturation — $k_1$

The fraction $\frac{f(k_1 + 1)}{f + k_1(\dots)}$ grows with $f$ but is **bounded**: as $f \to \infty$ the value approaches $k_1 + 1$, so extra occurrences yield diminishing returns. $k_1$ tunes how quickly saturation kicks in — typical values are **1.2–2.0**. At $k_1 = 0$ term frequency is ignored entirely (a term is either present or not).

### Length normalization — $b$

The term $\left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)$ scales the denominator by how long the document is relative to average. At $b = 0$ length is ignored; at $b = 1$ it is fully normalized. The default is **$b = 0.75$**. A longer-than-average document is penalized (its denominator grows), a shorter one is rewarded.

### Inverse document frequency

BM25 uses the **probabilistic IDF**, which differs from TF-IDF's:

$$\text{IDF}(t) = \log\left(1 + \frac{N - n_t + 0.5}{n_t + 0.5}\right)$$

where $N$ is the number of documents and $n_t$ is the number of documents containing $t$. The defining feature is the $N - n_t$ in the numerator: it penalizes common terms far more aggressively than TF-IDF's $\log(N/n_t)$. The $+0.5$ smoothing terms and the outer $+1$ keep the result finite and non-negative for every term.

### Parameters at a glance

| Parameter | Controls | Typical | Extremes |
|-----------|----------|---------|----------|
| $k_1$ | term-frequency saturation | 1.2–2.0 | $0$ = ignore TF; large = closer to linear TF |
| $b$ | document-length normalization | 0.75 | $0$ = ignore length; $1$ = full normalization |


In [1]:
import re
import numpy as np

In [ ]:
TOKEN = r'\w+(?:\.\w+)*'
    
class BM25:

    def __init__(self, k1, b, corpus):
        self.k1 = k1
        self.b = b
        self.corpus = corpus
        self.corpus_size = len(corpus)
        lengths = 0
        for doc in corpus:
            lengths += len(self.tokenize(doc))
        self.avg_corpus_length = lengths / self.corpus_size 

    def tokenize(self, text):
        return re.findall(TOKEN, text.lower())


    def calculate_tf(self, term, doc):
        tokens = self.tokenize(doc)
        if not tokens:
            return 0.0
        target = self.tokenize(term)
        if not target:
            return 0.0
        f_td = tokens.count(target[0])
        n = len(tokens)
        return (f_td * (self.k1 + 1)) / (f_td + (self.k1 * (1 - self.b + (self.b*(n / self.avg_corpus_length)))))

    def calculate_idf(self, term):
        target = self.tokenize(term)
        n = 0
        if self.corpus_size == 0:
            0
        for doc in self.corpus:
            d = self.tokenize(doc)
            if target[0] in d:
                n += 1
        return np.log((self.corpus_size + 1) / (n + 1)) + 1

    def calculate_bm25(self, term):
        idf = self.calculate_idf(term)
        tfs = []
        for doc in self.corpus:
            tf = self.calculate_tf(term, doc)
            tfs.append(tf)
        return np.array(tfs) * idf